# Phase 3: Reasoning Finetuning

This notebook fine-tunes Model H and Model L on the synthetic comparative reasoning datasets.

In [ ]:
!pip install -q huggingface_hub seaborn tqdm


In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download
import collections
import re
import string
import math
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)


In [ ]:
# Download Pretrained Checkpoints from HF
repo_id = "manyajain111/lma-phase2"
os.makedirs("checkpoints/pretrained/hindi", exist_ok=True)
os.makedirs("checkpoints/pretrained/nepali", exist_ok=True)

files_to_download = [
    ("hindi/model_final.pt", "checkpoints/pretrained/hindi"),
    ("hindi/config.json", "checkpoints/pretrained/hindi"),
    ("hindi/tokenizer.json", "checkpoints/pretrained/hindi"),
    ("nepali/model_final.pt", "checkpoints/pretrained/nepali"),
    ("nepali/config.json", "checkpoints/pretrained/nepali"),
    ("nepali/tokenizer.json", "checkpoints/pretrained/nepali")
]

for f, out_dir in files_to_download:
    if not os.path.exists(os.path.join(out_dir, os.path.basename(f))):
        print(f"Downloading {f}...")
        hf_hub_download(repo_id=repo_id, filename=f, local_dir="checkpoints/pretrained")
print("Downloads complete.")


In [ ]:
# --- INLINED DEPENDENCIES (Tokenizer and Model) ---

import json
import collections
import re
import string

PUNCT = re.escape(string.punctuation + "।॥‘’“”")
WORD_PAT = re.compile(rf" ?[^{PUNCT}\s]+| ?[{PUNCT}]+", re.UNICODE)

def detect_bpe_mode(vocab):
    """Identical to Phase 2's detect_bpe_mode: char-level tokens (Nepali) contain
    real Devanagari codepoints directly; byte-level tokens (Hindi) can't, since
    they're remapped to chr(0..255) before merges are learned."""
    for tok in vocab:
        if any(0x0900 <= ord(ch) <= 0x097F for ch in tok):
            return 'char'
    return 'byte'

class CustomBPETokenizer:
    def __init__(self, vocab_size=32000):
        self.vocab_size = vocab_size
        self.merges = []
        self.vocab = {}
        self.byte_to_char = {i: chr(i) for i in range(256)}
        self.char_to_byte = {v: k for k, v in self.byte_to_char.items()}
        self.merge_ranks = {}
        self.mode = 'byte'

    def _base_symbols(self, w):
        if self.mode == 'byte':
            return [self.byte_to_char[b] for b in w.encode('utf-8')]
        return list(w)

    def encode(self, text):
        if not hasattr(self, 'cache'):
            self.cache = {}
        ids = []
        for w in WORD_PAT.findall(text):
            if w not in self.cache:
                symbols = self._base_symbols(w)
                while len(symbols) > 1:
                    pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
                    ranked = [(self.merge_ranks[p], i) for i, p in enumerate(pairs) if p in self.merge_ranks]
                    if not ranked:
                        break
                    _, i = min(ranked)
                    symbols = symbols[:i] + [symbols[i] + symbols[i + 1]] + symbols[i + 2:]
                self.cache[w] = [self.vocab.get(s, 0) for s in symbols]
            ids.extend(self.cache[w])
        return ids

    def decode(self, ids):
        if getattr(self, 'id_to_tok', None) is None:
            self.id_to_tok = {v: k for k, v in self.vocab.items()}
        byte_arr = bytearray()
        for i in ids:
            for ch in self.id_to_tok.get(i, ""):
                try:
                    byte_arr.append(self.char_to_byte[ch])
                except KeyError:
                    byte_arr.extend(ch.encode("utf-8"))
        return bytes(byte_arr).decode("utf-8", errors="replace")

    def load(self, in_path):
        with open(in_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        self.vocab = data["vocab"]
        self.vocab_size = data.get("vocab_size", len(self.vocab))
        self.merges = [((a, b), a + b) for a, b in data["merges"]]
        self.merge_ranks = {pair: i for i, (pair, _) in enumerate(self.merges)}
        self.mode = detect_bpe_mode(self.vocab)

class TokenizerWrapper:
    def __init__(self, json_path):
        self.tokenizer = CustomBPETokenizer()
        self.tokenizer.load(json_path)

    def encode(self, text):
        return self.tokenizer.encode(text)

    def decode(self, ids):
        return self.tokenizer.decode(ids)

    @property
    def vocab_size(self):
        return self.tokenizer.vocab_size

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadCausalAttention(nn.Module):
    def __init__(self, d_model, n_heads, context_len, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.dk = d_model // n_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        
        mask = torch.triu(torch.ones(context_len, context_len) * float('-inf'), diagonal=1)
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, d_model = x.size()
        
        q = self.W_q(x).view(B, T, self.n_heads, self.dk).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_heads, self.dk).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_heads, self.dk).transpose(1, 2)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.dk)
        
        scores = scores + self.mask[:T, :T]
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.attn_drop(attn_weights)
        
        context = torch.matmul(attn_weights, v)
        
        context = context.transpose(1, 2).contiguous().view(B, T, d_model)
        
        out = self.resid_drop(self.W_o(context))
        return out, attn_weights

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        x = F.gelu(self.linear1(x))
        x = self.linear2(x)
        return self.dropout(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, context_len, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadCausalAttention(d_model, n_heads, context_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        
    def forward(self, x):
        attn_out, attn_weights = self.attn(self.ln1(x))
        x = x + attn_out
        
        x = x + self.ffn(self.ln2(x))
        return x, attn_weights

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, context_len, dropout=0.1, pad_token_id=0):
        super().__init__()
        self.context_len = context_len
        self.pad_token_id = pad_token_id
        
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_token_id)
        self.pos_emb = nn.Embedding(context_len, d_model)
        self.drop = nn.Dropout(dropout)
        
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, context_len, dropout)
            for _ in range(n_layers)
        ])
        
        self.ln_f = nn.LayerNorm(d_model)
        
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        
        self.head.weight = self.token_emb.weight
        
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, return_attention=False):
        B, T = idx.size()
        assert T <= self.context_len, f"Sequence length {T} exceeds context length {self.context_len}"
        
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        
        tok_emb = self.token_emb(idx)
        pos_emb = self.pos_emb(pos)
        x = self.drop(tok_emb + pos_emb)
        
        attention_maps = []
        for block in self.blocks:
            x, attn_weights = block(x)
            if return_attention:
                attention_maps.append(attn_weights)
                
        x = self.ln_f(x)
        logits = self.head(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
            
        if return_attention:
            return logits, loss, attention_maps
        return logits, loss

    @torch.no_grad()
    def greedy_generate(self, idx, max_new_tokens):
        """
        Greedy decoding: at each step, pick the single most probable next token (argmax).
        Deterministic — temperature has no effect.
        Required by Section 2.3 of the spec alongside temperature sampling.
        """
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.context_len else idx[:, -self.context_len:]
            logits, _ = self(idx_cond)
            idx_next = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Temperature sampling: higher temperature -> more diverse / random output.
        temperature=0.5  -> sharper distribution (more conservative)
        temperature=1.0  -> unmodified distribution
        temperature=1.5  -> flatter distribution (more creative / random)
        """
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.context_len else idx[:, -self.context_len:]
            
            logits, _ = self(idx_cond)
            
            logits = logits[:, -1, :] / temperature
            
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
                
            probs = F.softmax(logits, dim=-1)
            
            idx_next = torch.multinomial(probs, num_samples=1)
            
            idx = torch.cat((idx, idx_next), dim=1)
            
        return idx


In [ ]:
class ReasoningDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, max_len=256):
        self.data = []
        skipped = 0
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                d = json.loads(line)
                prompt, answer = d['prompt'], d['answer']
                full_text = f"{prompt}\nउत्तर: {answer}"
                tokens = tokenizer.encode(full_text)

                prompt_text = f"{prompt}\nउत्तर: "
                prompt_len = len(tokenizer.encode(prompt_text))

                if len(tokens) > max_len:
                    tokens = tokens[:max_len]

                input_ids = tokens[:-1]
                labels = tokens[1:]                       # shift by 1 for next-token prediction
                mask_len = max(prompt_len - 1, 0)
                for i in range(min(mask_len, len(labels))):
                    labels[i] = -100

                if len(labels) == 0 or all(l == -100 for l in labels):
                    skipped += 1
                    continue

                pad_len = (max_len - 1) - len(input_ids)
                if pad_len > 0:
                    input_ids = input_ids + [0] * pad_len
                    labels = labels + [-100] * pad_len

                self.data.append((torch.tensor(input_ids, dtype=torch.long),
                                   torch.tensor(labels, dtype=torch.long)))
        if skipped:
            print(f"Skipped {skipped}/{skipped + len(self.data)} examples in {jsonl_path} (answer fully truncated)")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

In [ ]:
def train_epoch(model, dataloader, optimizer, device, accum_steps=1):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    for i, (idx, targets) in enumerate(tqdm(dataloader, desc="Training")):
        idx, targets = idx.to(device), targets.to(device)

        logits, loss = model(idx, targets=targets)
        if loss.dim() > 0:
            loss = loss.mean()

        # Scale down so accumulated gradients match a full effective batch
        (loss / accum_steps).backward()

        if (i + 1) % accum_steps == 0 or (i + 1) == len(dataloader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate_loss(model, dataloader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for idx, targets in dataloader:
            idx, targets = idx.to(device), targets.to(device)
            logits, loss = model(idx, targets=targets)
            
            if loss.dim() > 0: 
                loss = loss.mean()
                
            total_loss += loss.item()
    return total_loss / len(dataloader)

In [ ]:
!ls /kaggle/input/datasets/manyajain111/lma-phase3-reasoning-data/


In [ ]:
# 1. Force delete the old 'repo' folder
!rm -rf repo

# 2. Clone it fresh (replace with your actual GitHub token if it's private!)
!git clone -b Phase-3 https://***REMOVED***@github.com/CL3-410/individual-project-ManyaJain111.git repo

# 3. Point the data directory to the newly cloned folder
DATA_DIR = "repo/phase3/data/"
print("done")

In [ ]:
import os
import csv
import torch
from torch.utils.data import DataLoader

def run_finetuning(lang_short, lang_full, epochs=3, batch_size=32, lr=5e-5, max_len=256, accum_steps=1):
    print(f"\n{'='*40}\nFinetuning {lang_full.upper()}\n{'='*40}")
    model, tokenizer = load_model_and_tokenizer(lang_full, f"checkpoints/pretrained/{lang_full}/model_final.pt")

    if max_len - 1 > model.context_len:
        max_len = model.context_len + 1

    train_ds = ReasoningDataset(os.path.join(DATA_DIR, f"{lang_short}/train.jsonl"), tokenizer, max_len=max_len)
    val_ds = ReasoningDataset(os.path.join(DATA_DIR, f"{lang_short}/val.jsonl"), tokenizer, max_len=max_len)

    micro_batch = max(1, batch_size // accum_steps)
    train_loader = DataLoader(train_ds, batch_size=micro_batch, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=micro_batch, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_loss = float('inf')
    out_dir = f"checkpoints/finetuned/{lang_full}"
    os.makedirs(out_dir, exist_ok=True)

    history = {'epoch': [], 'train_loss': [], 'val_loss': []}
    global_step = 0

    for epoch in range(epochs):
        train_loss = train_epoch(model, train_loader, optimizer, device, accum_steps=accum_steps)
        val_loss = evaluate_loss(model, val_loader, device)
        scheduler.step()
        global_step += len(train_loader)

        history['epoch'].append(epoch + 1)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        # RESUME-CAPABLE checkpoint (spec compliant: weights + optimizer + scheduler + step + config)
        epoch_ckpt = {
            'epoch': epoch + 1,
            'step': global_step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
        }
        epoch_ckpt_path = f"{out_dir}/model_finetuned_epoch_{epoch+1}.pt"
        torch.save(epoch_ckpt, epoch_ckpt_path)
        print(f"  -> Checkpoint for Epoch {epoch+1} saved to {epoch_ckpt_path}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_ckpt_path = f"{out_dir}/model_finetuned.pt"
            torch.save(epoch_ckpt, best_ckpt_path)
            print(f"  -> New best val_loss ({val_loss:.4f})! Saved to {best_ckpt_path}")

    # Save log to CSV
    csv_path = f"{out_dir}/finetune_log.csv"
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['epoch', 'train_loss', 'val_loss'])
        for ep, tl, vl in zip(history['epoch'], history['train_loss'], history['val_loss']):
            writer.writerow([ep, f"{tl:.4f}", f"{vl:.4f}"])
    print(f"Training log saved to {csv_path}")

    model.history = history
    print(f"\nTraining complete! Best val_loss={best_val_loss:.4f}. All {epochs} checkpoints saved in {out_dir}/")

    return model, tokenizer


In [ ]:
def load_model_and_tokenizer(lang_full_name, checkpoint_path):
    # Adjust this base_path if your checkpoints are inside the 'repo' folder
    base_path = f"checkpoints/pretrained/{lang_full_name}"
    
    with open(f"{base_path}/config.json", 'r') as f:
        config = json.load(f)
        
    tokenizer = TokenizerWrapper(f"{base_path}/tokenizer.json")
    
    model = GPTLanguageModel(
        vocab_size=config['vocab_size'],
        d_model=config['d_model'],
        n_heads=config['n_heads'],
        n_layers=config['n_layers'],
        d_ff=config['d_ff'],
        context_len=config['context_len'],
        dropout=0.1,
        pad_token_id=0
    )
    
    ckpt = torch.load(checkpoint_path, map_location='cpu')
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        state_dict = ckpt["model_state_dict"]
    else:
        state_dict = ckpt
        
    model.load_state_dict(state_dict)
    model.to(device)
    
    return model, tokenizer

In [ ]:
# 128 is perfect for Hindi (max_len 256 is short)
# accum_steps=4 -> actual micro-batch = 32, but effective batch stays 128 (same as before)
model_h_ft, tok_h = run_finetuning("hi", "hindi", epochs=3, lr=1e-5, batch_size=128, max_len=256, accum_steps=4)

# 32 is much safer for Nepali (max_len 768 takes 9x more memory per example!)
model_l_ft, tok_l = run_finetuning("ne", "nepali", epochs=3, lr=1e-5, batch_size=128, max_len=256, accum_steps=4)

In [ ]:
# Force load the best Epoch 1 checkpoint from the hard drive for Hindi
model_h_ft, tok_h = load_model_and_tokenizer("hindi", "checkpoints/finetuned/hindi/model_finetuned.pt")

# Force load the best checkpoint for Nepali (only works if it finished at least Epoch 1 before you hit stop!)
model_l_ft, tok_l = load_model_and_tokenizer("nepali", "checkpoints/finetuned/nepali/model_finetuned.pt")

In [ ]:
# Evaluation on Test Set (Exact Match by Tier)
def evaluate_exact_match(model, tokenizer, jsonl_path, num_samples=900):
    model.eval()
    correct_by_tier = {1: 0, 2: 0, 3: 0}
    total_by_tier = {1: 0, 2: 0, 3: 0}
    
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()[:num_samples]
        
    with torch.no_grad():
        for line in tqdm(lines, desc="Evaluating"):
            d = json.loads(line)
            prompt = d['prompt']
            label = d['label'] # Exact entity name
            tier = d.get('tier', 1)
            
            # Context
            context_str = f"{prompt}\nउत्तर: "
            idx = torch.tensor([tokenizer.encode(context_str)], dtype=torch.long).to(device)
            
            # Generate up to 100 tokens
            out_idx = model.greedy_generate(idx, max_new_tokens=100)
            generated_str = tokenizer.decode(out_idx[0].tolist())
            
            # Fix slicing bug: Decode the exact context tokens so lengths match exactly
            decoded_context = tokenizer.decode(tokenizer.encode(context_str))
            gen_answer = generated_str[len(decoded_context):]
            
            # --- TRUNCATION FIX ADDED HERE ---
            if '।' in gen_answer:
                gen_answer = gen_answer.split('।')[0] + '।'
            gen_answer = gen_answer.strip()
            # ---------------------------------
            
            if label in gen_answer:
                correct_by_tier[tier] += 1
            total_by_tier[tier] += 1
            
    accuracies = {}
    for t in [1, 2, 3]:
        if total_by_tier[t] > 0:
            accuracies[t] = correct_by_tier[t] / total_by_tier[t] * 100
        else:
            accuracies[t] = 0.0
            
    total_correct = sum(correct_by_tier.values())
    total = sum(total_by_tier.values())
    overall = total_correct / total * 100 if total > 0 else 0.0
    
    return overall, accuracies

print("\n--- EXACT MATCH ACCURACY ---")
try:
    print("Loading Pretrained Models for Baseline...")
    model_h_pre, _ = load_model_and_tokenizer("hindi", "checkpoints/pretrained/hindi/model_final.pt")
    model_l_pre, _ = load_model_and_tokenizer("nepali", "checkpoints/pretrained/nepali/model_final.pt")
    
    print("\nEvaluating Pretrained (Baseline)...")
    acc_h_pre, _ = evaluate_exact_match(model_h_pre, tok_h, os.path.join(DATA_DIR, "hi/test.jsonl"), num_samples=200)
    acc_l_pre, _ = evaluate_exact_match(model_l_pre, tok_l, os.path.join(DATA_DIR, "ne/test.jsonl"), num_samples=200)
    print(f"HINDI Pretrained Accuracy:  {acc_h_pre:.2f}%")
    print(f"NEPALI Pretrained Accuracy: {acc_l_pre:.2f}%")
    
    print("\nEvaluating Finetuned...")
    acc_h_ft, tiers_h = evaluate_exact_match(model_h_ft, tok_h, os.path.join(DATA_DIR, "hi/test.jsonl"), num_samples=500)
    acc_l_ft, tiers_l = evaluate_exact_match(model_l_ft, tok_l, os.path.join(DATA_DIR, "ne/test.jsonl"), num_samples=500)
    
    print(f"\n--- FINAL RESULTS ---")
    print(f"HINDI Overall Accuracy: {acc_h_ft:.2f}%")
    print(f"  Tier 1 (Direct):      {tiers_h[1]:.2f}%")
    print(f"  Tier 2 (Transitive):  {tiers_h[2]:.2f}%")
    print(f"  Tier 3 (Structural):  {tiers_h[3]:.2f}%")
    
    print(f"\nNEPALI Overall Accuracy: {acc_l_ft:.2f}%")
    print(f"  Tier 1 (Direct):      {tiers_l[1]:.2f}%")
    print(f"  Tier 2 (Transitive):  {tiers_l[2]:.2f}%")
    print(f"  Tier 3 (Structural):  {tiers_l[3]:.2f}%")
    
except Exception as e:
    print(f"Evaluation failed: {e}")
    



In [ ]:
import json
import torch

# Put model in evaluation mode
model_h_ft.eval()

print("--- 5 SAMPLE HINDI PREDICTIONS ---")

# Open the test set and grab the first 5 lines
test_file = os.path.join(DATA_DIR, "hi/test.jsonl")
with open(test_file, "r", encoding="utf-8") as f:
    lines = f.readlines()[:5]

for i, line in enumerate(lines):
    data = json.loads(line)
    prompt_text = data["prompt"]
    expected_answer = data["answer"]
    
    # 1. Encode the prompt
    input_ids = tok_h.encode(prompt_text)
    idx = torch.tensor([input_ids], dtype=torch.long).to(device)
    
    # 2. Generate response (max 20 new tokens is plenty for these short answers)
    with torch.no_grad():
        out_idx = model_h_ft.greedy_generate(idx, max_new_tokens=20)
        
    # 3. Decode only the NEWly generated tokens (ignoring the prompt)
    generated_tokens = out_idx[0].tolist()[len(input_ids):]
    prediction = tok_h.decode(generated_tokens)
    
    print(f"\n[Example {i+1}] (Tier {data['tier']} - {data['type']})")
    print(f"PROMPT:    {prompt_text}")
    print(f"EXPECTED:  {expected_answer}")
    print(f"PREDICTED: {prediction.strip()}")
    print("-" * 60)

In [ ]:
# Attention Visualization - Saves heatmaps to report/ directory
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs("report/hindi", exist_ok=True)
os.makedirs("report/nepali", exist_ok=True)

def plot_attention_heatmaps(model, tokenizer, text, layer_idx=0, head_idx=0, save_path=None, title_prefix=""):
    """Plot and optionally save attention heatmaps for a given layer and head."""
    model.eval()
    tokens = tokenizer.encode(text)
    idx = torch.tensor([tokens], dtype=torch.long).to(device)
    
    with torch.no_grad():
        if hasattr(model, 'module'):
            logits, _, attention_maps = model.module(idx, return_attention=True)
        else:
            logits, _, attention_maps = model(idx, return_attention=True)
    
    attn = attention_maps[layer_idx][0, head_idx].cpu().numpy()
    
    tok_strs = []
    for t in tokens:
        s = tokenizer.decode([t])
        tok_strs.append(s if s.strip() else f"<{t}>")
        
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(attn, xticklabels=tok_strs, yticklabels=tok_strs, cmap='viridis', ax=ax)
    ax.set_title(f"{title_prefix} Layer {layer_idx}, Head {head_idx}")
    ax.set_xlabel("Key Position")
    ax.set_ylabel("Query Position")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.close()

sample_hi = "यदि राम की आयु 25 साल है और श्याम की आयु 30 साल है, तो किसकी आयु कम है?\nउत्तर: राम"
sample_ne = "यदि राम सँग 85 रुपैयाँ छ र सीता सँग 42 रुपैयाँ छ। को सँग थोरै रुपैयाँ छ?\nउत्तर: सीता"

try:
    # Hindi: early (layer 0) and late (layer 5) post-finetune
    plot_attention_heatmaps(model_h_ft, tok_h, sample_hi, layer_idx=0, head_idx=0,
        save_path="report/hindi/attn_finetuned_layer0.png", title_prefix="Hindi Finetuned")
    plot_attention_heatmaps(model_h_ft, tok_h, sample_hi, layer_idx=5, head_idx=0,
        save_path="report/hindi/attn_finetuned_layer5.png", title_prefix="Hindi Finetuned")
    print("Hindi finetuned attention heatmaps saved.")
except Exception as e:
    print(f"Hindi attention failed: {e}")

try:
    # Nepali: early and late post-finetune
    plot_attention_heatmaps(model_l_ft, tok_l, sample_ne, layer_idx=0, head_idx=0,
        save_path="report/nepali/attn_finetuned_layer0.png", title_prefix="Nepali Finetuned")
    plot_attention_heatmaps(model_l_ft, tok_l, sample_ne, layer_idx=5, head_idx=0,
        save_path="report/nepali/attn_finetuned_layer5.png", title_prefix="Nepali Finetuned")
    print("Nepali finetuned attention heatmaps saved.")
except Exception as e:
    print(f"Nepali attention failed: {e}")


In [ ]:
tokenizer_ne = TokenizerWrapper("checkpoints/pretrained/nepali/tokenizer.json")

lengths = []
with open(os.path.join(DATA_DIR, "ne/train.jsonl"), encoding='utf-8') as f:
    for line in f:
        d = json.loads(line)
        full = f"{d['prompt']}\nउत्तर: {d['answer']}"
        lengths.append(len(tokenizer_ne.encode(full)))
print("min/median/max:", min(lengths), sorted(lengths)[len(lengths)//2], max(lengths))

In [ ]:
# --- VERIFICATION --- 
# Run this to verify the mode was detected properly and tokenizer works.
try:
    print("Detected mode (Nepali):", tok_l.tokenizer.mode)  # must print 'char' for Nepali
    test = "नेपाल एउटा सुन्दर देश हो।"
    print("Round-trip:", tok_l.decode(tok_l.encode(test)))  # must exactly match `test`
except NameError:
    print("Load models first!")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs("report/phase3_plots", exist_ok=True)

def plot_attention_comparison(model_pre, model_ft, tokenizer, text, layer_idx, lang_label, save_path):
    """Side-by-side pretrained vs finetuned attention heatmap for a given layer. Saves PNG."""
    tokens = tokenizer.encode(text)
    decoded_tokens = [tokenizer.decode([t]) if tokenizer.decode([t]).strip() else f"<{t}>" for t in tokens]
    idx = torch.tensor([tokens], dtype=torch.long).to(device)
    
    with torch.no_grad():
        m = model_pre.module if hasattr(model_pre, 'module') else model_pre
        _, _, attn_pre = m(idx, return_attention=True)
        m = model_ft.module if hasattr(model_ft, 'module') else model_ft
        _, _, attn_ft = m(idx, return_attention=True)
        
    map_pre = attn_pre[layer_idx][0, 0].cpu().numpy()
    map_ft  = attn_ft[layer_idx][0, 0].cpu().numpy()
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    sns.heatmap(map_pre, xticklabels=decoded_tokens, yticklabels=decoded_tokens,
                cmap="viridis", ax=axes[0], cbar=True)
    axes[0].set_title(f"{lang_label} — Pretrained (Layer {layer_idx})", fontsize=13)
    axes[0].set_xlabel("Key Position"); axes[0].set_ylabel("Query Position")
    
    sns.heatmap(map_ft, xticklabels=decoded_tokens, yticklabels=decoded_tokens,
                cmap="viridis", ax=axes[1], cbar=True)
    axes[1].set_title(f"{lang_label} — Finetuned (Layer {layer_idx})", fontsize=13)
    axes[1].set_xlabel("Key Position"); axes[1].set_ylabel("Query Position")
    
    plt.suptitle(f"{lang_label}: Pre- vs Post-Finetuning Attention (Layer {layer_idx})", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"Saved: {save_path}")
    plt.close()

sample_hi = "यदि राम की आयु 25 साल है और श्याम की आयु 30 साल है, तो किसकी आयु कम है?\nउत्तर: राम"
sample_ne = "यदि राम सँग 85 रुपैयाँ छ र सीता सँग 42 रुपैयाँ छ। को सँग थोरै रुपैयाँ छ?\nउत्तर: सीता"

# Load pretrained models fresh for comparison
model_h_pre, _ = load_model_and_tokenizer("hindi", "checkpoints/pretrained/hindi/model_final.pt")
model_l_pre, _ = load_model_and_tokenizer("nepali", "checkpoints/pretrained/nepali/model_final.pt")
model_h_pre.eval(); model_h_ft.eval()
model_l_pre.eval(); model_l_ft.eval()

# Hindi: early (0) and late (5) layers
plot_attention_comparison(model_h_pre, model_h_ft, tok_h, sample_hi, layer_idx=0,
    lang_label="Hindi", save_path="report/phase3_plots/hindi_attn_compare_layer0.png")
plot_attention_comparison(model_h_pre, model_h_ft, tok_h, sample_hi, layer_idx=5,
    lang_label="Hindi", save_path="report/phase3_plots/hindi_attn_compare_layer5.png")

# Nepali: early (0) and late (5) layers  
plot_attention_comparison(model_l_pre, model_l_ft, tok_l, sample_ne, layer_idx=0,
    lang_label="Nepali", save_path="report/phase3_plots/nepali_attn_compare_layer0.png")
plot_attention_comparison(model_l_pre, model_l_ft, tok_l, sample_ne, layer_idx=5,
    lang_label="Nepali", save_path="report/phase3_plots/nepali_attn_compare_layer5.png")

print("\n✅ All pre vs post attention comparison plots saved to report/phase3_plots/")


In [ ]:
!cd repo && git pull origin Phase-3

In [ ]:
import json
import torch

model_h_ft.eval()
print("--- TESTING ON 60 ULTRA-SIMPLE PROMPTS ---")

correct = 0
total = 0

# Reading the dataset we just pulled from GitHub
with open("repo/phase3/data/simple/test_easy.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        prompt_text = data["prompt"]
        label = data["label"]
        
        # Add the "उत्तर: " (Answer:) tag
        context_str = f"{prompt_text}\nउत्तर: "
        
        input_ids = tok_h.encode(context_str)
        idx = torch.tensor([input_ids], dtype=torch.long).to(device)
        
        with torch.no_grad():
            # Handle DataParallel if it's wrapped
            if hasattr(model_h_ft, 'module'):
                out_idx = model_h_ft.module.greedy_generate(idx, max_new_tokens=20)
            else:
                out_idx = model_h_ft.greedy_generate(idx, max_new_tokens=20)
            
        decoded_context = tok_h.decode(input_ids)
        generated_str = tok_h.decode(out_idx[0].tolist())
        gen_answer = generated_str[len(decoded_context):]
        
        # Truncate at the first full stop
        if '।' in gen_answer:
            gen_answer = gen_answer.split('।')[0] + '।'
        gen_answer = gen_answer.strip()
        
        if label in gen_answer:
            correct += 1
        total += 1
        
print(f"\nEASY PROMPT ACCURACY: {correct/total * 100:.2f}%")

In [ ]:
# Per-Type Accuracy Breakdown (for grading)
import collections, json, os

def evaluate_per_type(model, tokenizer, jsonl_path, num_samples=1350):
    """Evaluate exact-match accuracy grouped by reasoning type."""
    model.eval()
    correct_by_type = collections.defaultdict(int)
    total_by_type   = collections.defaultdict(int)
    
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()[:num_samples]
        
    with torch.no_grad():
        for line in tqdm(lines, desc="Per-type eval"):
            d = json.loads(line)
            prompt = d['prompt']
            label  = d['label']
            qtype  = d.get('type', 'unknown')
            
            context_str = f"{prompt}\nउत्तर: "
            idx = torch.tensor([tokenizer.encode(context_str)], dtype=torch.long).to(device)
            out_idx = model.greedy_generate(idx, max_new_tokens=80)
            generated_str = tokenizer.decode(out_idx[0].tolist())
            decoded_context = tokenizer.decode(tokenizer.encode(context_str))
            gen_answer = generated_str[len(decoded_context):]
            if '।' in gen_answer:
                gen_answer = gen_answer.split('।')[0]
            gen_answer = gen_answer.strip()
            
            if label in gen_answer:
                correct_by_type[qtype] += 1
            total_by_type[qtype] += 1
    
    print(f"\n{'Type':<30} {'Correct':>7} {'Total':>7} {'Acc%':>7}")
    print("-" * 55)
    for qtype in sorted(total_by_type):
        c, t = correct_by_type[qtype], total_by_type[qtype]
        print(f"{qtype:<30} {c:>7} {t:>7} {100*c/t:>6.1f}%")

print("=== HINDI Per-Type Accuracy ===")
try:
    evaluate_per_type(model_h_ft, tok_h, os.path.join(DATA_DIR, "hi/test.jsonl"))
except Exception as e:
    print(f"Failed: {e}")

print("\n=== NEPALI Per-Type Accuracy ===")
try:
    evaluate_per_type(model_l_ft, tok_l, os.path.join(DATA_DIR, "ne/test.jsonl"))
except Exception as e:
    print(f"Failed: {e}")


In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

def plot_finetuning_curves():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    langs = [("hindi", "Model H (Hindi)", 0), ("nepali", "Model L (Nepali)", 1)]
    
    for lang_full, title, ax_idx in langs:
        ax = axes[ax_idx]
        epochs, train_losses, val_losses = [], [], []
        
        # 1. Try reading from model in memory
        try:
            model_obj = model_h_ft if lang_full == 'hindi' else model_l_ft
            if hasattr(model_obj, 'history') and model_obj.history:
                epochs = model_obj.history['epoch']
                train_losses = [float(x) for x in model_obj.history['train_loss']]
                val_losses = [float(x) for x in model_obj.history['val_loss']]
        except NameError:
            pass
            
        # 2. Fall back to reading saved CSV log
        if not epochs:
            csv_path = f"checkpoints/finetuned/{lang_full}/finetune_log.csv"
            if os.path.exists(csv_path):
                df = pd.read_csv(csv_path)
                epochs = df['epoch'].tolist()
                train_losses = df['train_loss'].tolist()
                val_losses = df['val_loss'].tolist()
                
        if epochs:
            ax.plot(epochs, train_losses, marker='o', linewidth=2.2, color='#1f77b4', label='Training Loss')
            ax.plot(epochs, val_losses, marker='s', linewidth=2.2, color='#d62728', linestyle='--', label='Validation Loss')
            
            min_val = min(val_losses)
            min_idx = val_losses.index(min_val)
            best_epoch = epochs[min_idx]
            ax.scatter([best_epoch], [min_val], color='#2ca02c', s=130, zorder=5, label=f'Best Val ({min_val:.4f})')
            ax.annotate(f'Best: {min_val:.4f}\n(Epoch {best_epoch})', 
                        xy=(best_epoch, min_val),
                        xytext=(best_epoch + 0.15, min_val + 0.02),
                        fontweight='bold', color='#2ca02c')
            
            ax.set_title(f"{title} — Reasoning Finetuning", fontsize=12, fontweight='bold', pad=10)
            ax.set_xlabel("Epoch", fontsize=11)
            ax.set_ylabel("Cross-Entropy Loss", fontsize=11)
            ax.set_xticks(epochs)
            ax.grid(True, linestyle=':', alpha=0.6)
            ax.legend(frameon=True, fontsize=10)
        else:
            ax.text(0.5, 0.5, f"No log data found for {title}\n(Run training cell first)", 
                    ha='center', va='center', transform=ax.transAxes, color='gray')
            ax.set_title(title)
            
    plt.tight_layout()
    os.makedirs("report", exist_ok=True)
    plt.savefig("report/finetuning_loss_curves.png", dpi=300, bbox_inches="tight")
    print("Plot saved to report/finetuning_loss_curves.png")
    plt.show()

plot_finetuning_curves()